In [2]:
import sys, os
from dotenv import load_dotenv
import pandas, pyoxigraph

load_dotenv()
print("Python:", sys.executable)          # deve contenere .venv
print("pandas:", pandas.__version__, "| pyoxigraph:", pyoxigraph.__version__)
print("Chiave presente:", bool(os.getenv("GOOGLE_API_KEY")))   # deve stampare True

Python: /home/piopo/Work/MasterDegree/SecondMaster/GDB/progetto/.venv/bin/python
pandas: 3.1.0 | pyoxigraph: 0.5.11
Chiave presente: False


In [4]:
import httpx
spec = httpx.get("http://localhost:8889/openapi.json", timeout=10).json()
print("Schemi di sicurezza:", spec.get("components", {}).get("securitySchemes"))
print()
for path, methods in spec["paths"].items():
    if any(k in path.lower() for k in ["auth", "key", "token", "login", "v1", "chat", "inference", "model"]):
        print(path, list(methods.keys()))

Schemi di sicurezza: {'HTTPBearer': {'type': 'http', 'scheme': 'bearer'}}

/api/auth/identity ['get']
/api/auth/status ['get']
/api/auth/login ['post']
/api/auth/logout ['post']
/api/auth/desktop-login ['post']
/api/auth/refresh ['post']
/api/auth/desktop-initial-password ['post']
/api/auth/change-password ['post']
/api/auth/api-keys ['get', 'post']
/api/auth/api-keys/{key_id} ['delete']
/api/accounts/owner/login-alias ['put']
/api/accounts/{account_id}/model-access ['put']
/api/train/token-preview ['post']
/api/models/local ['get']
/api/models/scan-folders ['get', 'post']
/api/models/scan-folders/{folder_id} ['delete']
/api/models/recommended-folders ['get']
/api/models/browse-folders ['get']
/api/models/list ['get']
/api/models/config/{model_name} ['get']
/api/models/remote-code-scan ['post']
/api/models/discard-remote-code ['post']
/api/models/loras ['get']
/api/models/diffusion-loras ['get']
/api/models/diffusion-embeddings ['get']
/api/models/diffusion-controlnets ['get']
/api/mod

In [9]:
import os, json, httpx
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
URL = os.getenv("LOCAL_MODEL_URL")
H = {"Authorization": f"Bearer {os.getenv('LOCAL_MODEL_API_KEY')}"}
MODEL = "unsloth/gemma-4-E2B-it-GGUF"

schema = {"name": "answer", "schema": {"type": "object",
          "properties": {"answer": {"type": "boolean"}}, "required": ["answer"]}}

casi = {
    "positivo (atteso true)": "Super Mario Bros. is a 1985 side-scrolling game where the player jumps across platforms.",
    "negativo (atteso false)": "Gran Turismo is a 1997 racing game where the player drives licensed cars on circuits.",
}
for nome, testo in casi.items():
    body = {"model": MODEL, "temperature": 0, "max_tokens": 300,
            "messages": [{"role": "user", "content": f"Criterion: the game is a platformer\nInput: {testo}"}],
            "response_format": {"type": "json_schema", "json_schema": schema}}
    r = httpx.post(URL + "/chat/completions", json=body, headers=H, timeout=120)
    print(nome, "->", r.status_code)
    print("   ", r.json()["choices"][0]["message"]["content"] if r.status_code == 200 else r.text[:300])

positivo (atteso true) -> 200
    {
  "answer": true
}
negativo (atteso false) -> 200
    {
  "answer": false
}


In [1]:
import os, httpx
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
key = os.getenv("GOOGLE_API_KEY")
r = httpx.get("https://generativelanguage.googleapis.com/v1beta/models",
              headers={"x-goog-api-key": key}, timeout=30)
print(r.status_code)
print([m["name"] for m in r.json().get("models", [])][:15] if r.status_code == 200 else r.text[:300])

200
['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview']


In [3]:
r = httpx.get("https://generativelanguage.googleapis.com/v1beta/models?pageSize=200",
              headers={"x-goog-api-key": key}, timeout=30)
for m in r.json().get("models", []):
    if "flash" in m["name"] and "generateContent" in m.get("supportedGenerationMethods", []):
        print(m["name"])

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts


In [4]:
import os, httpx
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
key = os.getenv("GOOGLE_API_KEY")
MODEL = "gemini-3.1-flash-lite"

body = {"model": MODEL, "temperature": 0, "max_tokens": 300,
        "messages": [{"role": "user", "content":
            "Criterion: the game is a platformer\n"
            "Input: Super Mario Bros. is a 1985 side-scrolling game where the player jumps across platforms."}],
        "response_format": {"type": "json_schema", "json_schema": {
            "name": "answer", "schema": {"type": "object",
            "properties": {"answer": {"type": "boolean"}}, "required": ["answer"]}}}}
r = httpx.post("https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
               json=body, headers={"Authorization": f"Bearer {key}"}, timeout=60)
print(r.status_code)
print(r.json()["choices"][0]["message"]["content"] if r.status_code == 200 else r.text[:300])

200
{
  "answer": true
}
